# Olist: phân khúc khách bằng RFM + k-means

Đội CRM muốn chia khách thành vài nhóm để chọn chiến dịch (win-back, chăm sóc khách giá trị cao…). Cách kinh
điển: tính **RFM** cho mỗi khách rồi phân cụm.

- **Recency**: số ngày trọn từ lần mua gần nhất tới ngày chốt (snapshot) 2018-09-01.
- **Frequency**: số đơn giao xong.
- **Monetary**: tổng `payment_value` của các đơn đó.

Khách là `customer_unique_id` (Olist cấp `customer_id` mới cho **mỗi đơn**; dùng nhầm `customer_id` thì
ai cũng chỉ mua một lần). Đơn giao xong: `order_status == "delivered"` và có `order_delivered_customer_date`,
mua trước snapshot.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật, phân cụm, vẽ. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/11-classic-ml/exercises/intermediate/03-olist-rfm-segments -q`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

## 1. Bảng RFM

`rfm_table(orders, customers, payments, snapshot)`: index `customer_unique_id`, cột theo thứ tự
`recency` (int, số ngày trọn, làm tròn xuống: `(snapshot − lần mua cuối).days`), `frequency` (số đơn), `monetary` (tổng
`payment_value`; một đơn có thể có nhiều dòng payment; đơn không có payment nào tính 0).

In [ ]:
def rfm_table(orders: pd.DataFrame, customers: pd.DataFrame, payments: pd.DataFrame,
              snapshot: pd.Timestamp) -> pd.DataFrame:
    # TODO: lọc đơn giao xong, mua trước snapshot
    # TODO: gắn customer_unique_id, cộng payment theo đơn (left join, thiếu thì 0)
    # TODO: groupby customer_unique_id
    raise NotImplementedError

In [ ]:
# explore: đọc Olist thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
orders = pd.read_csv(RAW / "olist_orders_dataset.csv",
                     parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"])
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
rfm = rfm_table(orders, customers, payments, pd.Timestamp("2018-09-01"))
print(len(rfm), "khách; mua từ 2 đơn trở lên:", round((rfm.frequency > 1).mean(), 4))
rfm.describe().round(1)

Nhìn `frequency`: phần lớn khách chỉ mua một lần. Điều đó nói gì về chữ F trong RFM ở Olist?

## 2. Chuẩn bị feature và phân cụm

k-means dùng khoảng cách Euclid, nên thang đo quyết định tất cả. `monetary` lệch phải nặng (vài khách chi
hàng nghìn R$). `prepare_features(rfm)`: `log1p` cả ba cột rồi chuẩn hoá về trung bình 0, độ lệch chuẩn 1
(`StandardScaler`), trả `np.ndarray` cỡ `(n, 3)`.

`fit_kmeans(X, k, seed)`: nhãn cụm từ `KMeans(n_clusters=k, n_init=10, random_state=seed)`.

`silhouette_by_k(X, ks, seed, sample_size=10_000, n_init=3)`: `pd.Series` index là k, giá trị là silhouette
score của `KMeans(n_clusters=k, n_init=n_init, random_state=seed)` (tính trên mẫu `sample_size` điểm với
`random_state=seed` khi dữ liệu lớn hơn, vì silhouette tốn O(n²)). `n_init=3` thay vì 10 cho nhanh: chỉ để
so các k với nhau.

In [ ]:
def prepare_features(rfm: pd.DataFrame) -> np.ndarray:
    # TODO
    raise NotImplementedError


def fit_kmeans(X: np.ndarray, k: int, seed: int = 0) -> np.ndarray:
    # TODO
    raise NotImplementedError


def silhouette_by_k(X: np.ndarray, ks, seed: int = 0, sample_size: int = 10_000, n_init: int = 3) -> pd.Series:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: không scale thì k-means chỉ nhìn monetary
raw_labels = fit_kmeans(rfm[["recency", "frequency", "monetary"]].to_numpy(), 4)
print(rfm.groupby(raw_labels).agg(n=("monetary", "size"), monetary_min=("monetary", "min"),
                                  monetary_max=("monetary", "max"), recency_median=("recency", "median")))

In [ ]:
# explore: silhouette theo k (vài giây)
X = prepare_features(rfm)
scores = silhouette_by_k(X, range(2, 9))
scores.round(3)

## 3. Mô tả từng cụm

`segment_profile(rfm, labels)`: index là nhãn cụm (tăng dần), cột `n`, `share` (tỉ lệ khách),
`recency_median`, `frequency_mean`, `repeat_rate` (tỉ lệ khách có `frequency > 1`), `monetary_median`.

In [ ]:
def segment_profile(rfm: pd.DataFrame, labels) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: k = 2 (silhouette cao nhất) tách cái gì?
segment_profile(rfm, fit_kmeans(X, 2)).round(3)

In [ ]:
# explore: k = 4 cho CRM
labels4 = fit_kmeans(X, 4)
segment_profile(rfm, labels4).round(3)

In [ ]:
# explore: PCA chiếu 3 chiều xuống 2 để nhìn cụm (chỉ để xem, không dùng để phân cụm)
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

pca = PCA(n_components=2).fit(X)
Z = pca.transform(X)
print("explained variance ratio:", pca.explained_variance_ratio_.round(3))
idx = np.random.default_rng(0).choice(len(Z), 5000, replace=False)
plt.scatter(Z[idx, 0], Z[idx, 1], c=labels4[idx], s=4, cmap="tab10")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

In [ ]:
# explore: DBSCAN trên mẫu 10 000 khách: không cần chọn k, nhưng phải chọn eps và min_samples
from sklearn.cluster import DBSCAN

sample = X[np.random.default_rng(0).choice(len(X), 10_000, replace=False)]
for eps in [0.2, 0.3, 0.5]:
    db = DBSCAN(eps=eps, min_samples=20).fit(sample)
    n_clusters = len(set(db.labels_)) - (-1 in db.labels_)
    print(f"eps={eps}: {n_clusters} cụm, nhiễu {np.mean(db.labels_ == -1):.3f}")

## 4. Câu hỏi để tự trả lời

1. Silhouette cao nhất ở k = 2. Cụm đó tách khách theo tiêu chí gì? Bạn có cần k-means để làm việc đó không?
2. Đặt tên cho 4 cụm (ví dụ "mới mua, giá trị vừa", "lâu không quay lại, chi ít") và đề xuất một hành động
   CRM cho mỗi cụm. Cụm nào đáng tiền nhất để chăm sóc?
3. Với 97% khách mua một lần, phân cụm RFM có phải công cụ tốt nhất không? Bạn sẽ thử gì khác (quy tắc RFM
   theo ngũ phân vị, model xác suất quay lại, feature về sản phẩm, vùng)?
4. Chạy lại `fit_kmeans` với seed khác: cụm có ổn định không? Vì sao điều đó quan trọng khi đội CRM dùng
   nhãn cụm hằng tháng?